# KNN-Based Recommendation (Surprise `KNNBaseline`)

Surprise's `KNNBaseline` extends classical k-NN CF by first estimating a baseline rating `b_{ui} = μ + b_u + b_i` via Alternating Least Squares, subtracting it, then aggregating residuals across neighbors using a similarity-weighted average:

$$ \hat{r}_{ui} = b_{ui} + \frac{\sum_{j \in N^k_i(u)} \text{sim}(i, j) (r_{uj} - b_{uj})}{\sum_{j \in N^k_i(u)} \text{sim}(i, j)} $$

Reference: Koren (2010), *Factor in the Neighbors*.

We run **two variants** — item-based and user-based — with Pearson similarity and `k = 20` neighbors. The item-based variant typically wins on MovieLens because items have many more raters than users have rated items.


In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [2]:
# scikit-surprise 1.1.4 was compiled against NumPy 1.x. Colab's default is NumPy 2.x,
# which is binary-incompatible. Install numpy<2 first, then force a kernel restart so
# Python re-loads with the downgraded numpy. After the restart, re-run this cell —
# the pip install will be a no-op and the kernel-restart check will pass.
!pip install -q "numpy<2" scikit-surprise==1.1.4

import numpy as np
if np.__version__.startswith('2.'):
    print('Downgrading numpy required a kernel restart. Restarting now...')
    print('After restart: re-mount Drive (re-run the drive.mount cell) and re-run this cell.')
    import os
    os.kill(os.getpid(), 9)
else:
    print(f'NumPy {np.__version__} OK — scikit-surprise ready to import')

NumPy 1.26.4 OK — scikit-surprise ready to import


In [3]:
import os, random, sys
import numpy as np
import pandas as pd
from pathlib import Path
from surprise import KNNBaseline, Dataset, Reader

SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

ARTIFACT_DIR    = Path('/content/gdrive/MyDrive/recsys_artifacts')
SPLITS_DIR      = ARTIFACT_DIR / 'splits'
PREDICTIONS_DIR = ARTIFACT_DIR / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(ARTIFACT_DIR))
from recsys_metrics import build_candidate_lists, evaluate_model

In [4]:
train = pd.read_csv(SPLITS_DIR / 'train.csv')
val   = pd.read_csv(SPLITS_DIR / 'val.csv')
test  = pd.read_csv(SPLITS_DIR / 'test.csv')
trainval = pd.concat([train, val], ignore_index=True)
print(f'train: {len(train):,}  val: {len(val):,}  test: {len(test):,}')

train: 99,616  val: 610  test: 610


In [5]:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(trainval[['userId', 'movieId', 'rating']], reader)
trainset = data.build_full_trainset()

## Train both KNN variants

In [6]:
knn_item = KNNBaseline(k=20, sim_options={'name': 'pearson_baseline', 'user_based': False})
knn_item.fit(trainset)

Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.


In [7]:
knn_user = KNNBaseline(k=20, sim_options={'name': 'pearson_baseline', 'user_based': True})
knn_user.fit(trainset)

Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.


## Evaluate both variants

In [8]:
def make_predict_fns(algo):
    def predict_pointwise(df):
        out = df[['userId', 'movieId', 'rating']].copy()
        out['predicted_rating'] = [algo.predict(uid, iid).est for uid, iid in zip(df['userId'], df['movieId'])]
        return out
    def predict_fn(user_id, movie_ids):
        return np.array([algo.predict(int(user_id), int(m)).est for m in movie_ids], dtype=float)
    return predict_pointwise, predict_fn

candidates = build_candidate_lists(train, test, num_negatives=99, seed=SEED)

In [9]:
pw_item, pr_item = make_predict_fns(knn_item)
pointwise_item = pw_item(test)
metrics_item = evaluate_model(pr_item, test, candidates, pointwise_predictions=pointwise_item, k=10, threshold=3.5)
print('=== KNNBaseline (item-based) — test set ===')
for key in ['rmse', 'mae', 'hr_at_k', 'ndcg_at_k', 'precision_at_k', 'recall_at_k', 'f1_at_k']:
    print(f'  {key:18s} = {metrics_item[key]:.4f}')

=== KNNBaseline (item-based) — test set ===
  rmse               = 0.9691
  mae                = 0.7411
  hr_at_k            = 0.3148
  ndcg_at_k          = 0.1754
  precision_at_k     = 0.0269
  recall_at_k        = 0.2689
  f1_at_k            = 0.0489


In [10]:
pw_user, pr_user = make_predict_fns(knn_user)
pointwise_user = pw_user(test)
metrics_user = evaluate_model(pr_user, test, candidates, pointwise_predictions=pointwise_user, k=10, threshold=3.5)
print('=== KNNBaseline (user-based) — test set ===')
for key in ['rmse', 'mae', 'hr_at_k', 'ndcg_at_k', 'precision_at_k', 'recall_at_k', 'f1_at_k']:
    print(f'  {key:18s} = {metrics_user[key]:.4f}')

=== KNNBaseline (user-based) — test set ===
  rmse               = 0.9775
  mae                = 0.7487
  hr_at_k            = 0.1230
  ndcg_at_k          = 0.0513
  precision_at_k     = 0.0113
  recall_at_k        = 0.1131
  f1_at_k            = 0.0206


In [11]:
pointwise_item[['userId', 'movieId', 'predicted_rating']].to_csv(PREDICTIONS_DIR / 'knn_item.csv', index=False)
pointwise_user[['userId', 'movieId', 'predicted_rating']].to_csv(PREDICTIONS_DIR / 'knn_user.csv', index=False)
print(f'Saved item-KNN predictions -> {PREDICTIONS_DIR / "knn_item.csv"}')
print(f'Saved user-KNN predictions -> {PREDICTIONS_DIR / "knn_user.csv"}')

Saved item-KNN predictions -> /content/gdrive/MyDrive/recsys_artifacts/predictions/knn_item.csv
Saved user-KNN predictions -> /content/gdrive/MyDrive/recsys_artifacts/predictions/knn_user.csv
